# A Feeder from Scratch

This notebook builds a small three-phase distribution feeder with no source file: a 115 kV grid source, a
115/12.47 kV substation transformer, a feeder breaker, a two-mile overhead line, and a load, capacitor bank
and PV system at the end of the line. It then writes the model to CIM XML and reads it back.

```mermaid
flowchart LR
    grid(["EnergySource<br/>src_grid"]) --- sourcebus["sourcebus<br/>115 kV"]
    sourcebus --- xf["PowerTransformer<br/>xf_sub"] --- subbus["subbus<br/>12.47 kV"]
    subbus --- brk["Breaker<br/>brk_feeder"] --- head["feeder_head"]
    head --- line["ACLineSegment<br/>line_1, 2 mi"] --- bus1["bus_1"]
    bus1 --- load(["EnergyConsumer<br/>ld_1"])
    bus1 --- cap(["LinearShuntCompensator<br/>cap_1"])
    bus1 --- pv(["PowerElectronicsConnection<br/>pv_1"])
```

## Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
anything from CIMantic Graphs.

In [1]:
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"

import cimprofiles.cim18.cim18v17 as cim
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
from cimgraph.utils import write_xml
from cimtables.units import Qty

from cimbuilder.network import new_feeder_model
from cimbuilder.object_builder import (
    ACLineSegmentBuilder,
    BaseVoltageBuilder,
    BreakerBuilder,
    ConnectivityNodeBuilder,
    EnergyConsumerBuilder,
    EnergySourceBuilder,
    LinearShuntCompensatorBuilder,
    PerLengthSequenceImpedanceBuilder,
    PhotoVoltaicUnitBuilder,
    PowerElectronicsConnectionBuilder,
    PowerTransformerBuilder,
)

## The model, base voltages and nodes

`new_feeder_model` makes an empty model around the Feeder `fdr_demo`. Every piece of equipment below goes
in that Feeder unless it is given another `EquipmentContainer`.

Each voltage level gets a `BaseVoltage`, and each node is built up front, so that it has a name and a
`BaseVoltage`. A node that does not exist yet is created by the first equipment that connects to it, but
without either.

In [2]:
network = new_feeder_model(
    authority="https://example.com/models",
    model="feeder_from_scratch",
    Feeder="fdr_demo",
    name="demo",
)
container = network.container

base_voltages = BaseVoltageBuilder(network, container)
base_voltages.create(mRID="bv_115kV", name="115 kV", nominalVoltage=Qty(115, "kV"))
base_voltages.create(
    mRID="bv_12_47kV", name="12.47 kV", nominalVoltage=Qty(12.47, "kV")
)

node_builder = ConnectivityNodeBuilder(network, container)
for mRID, base_voltage in [
    ("sourcebus", "bv_115kV"),
    ("subbus", "bv_12_47kV"),
    ("feeder_head", "bv_12_47kV"),
    ("bus_1", "bv_12_47kV"),
]:
    node = node_builder.create(mRID=mRID, name=mRID)
    node_builder.add_connectivity(node, BaseVoltage=base_voltage)

## The grid source

An `EnergySource` is the Thevenin equivalent of the grid at `sourcebus`: a voltage behind an impedance.
`phases="ABC"` gives it one `EnergySourcePhase` per phase.

In [3]:
source_builder = EnergySourceBuilder(network, container)
source = source_builder.create(mRID="src_grid", name="grid")
source, terminal, source_phases = source_builder.add_connectivity(
    source,
    node="sourcebus",
    nominalVoltage=Qty(115, "kV"),
    phases="ABC",
    BaseVoltage="bv_115kV",
)
source_builder.add_electrical(source, r=Qty(0.5, "ohm"), x=Qty(5, "ohm"))
source_builder.add_ssh(
    source, voltageMagnitude=Qty(116.15, "kV"), voltageAngle=Qty(0, "deg")
)

print(terminal.mRID, [phase.mRID for phase in source_phases])

src_grid_T1 ['src_grid_A', 'src_grid_B', 'src_grid_C']


## The substation transformer

A `PowerTransformer` has one `PowerTransformerEnd` per winding, each with its own Terminal, node and
`BaseVoltage`. End 1 is the high-voltage winding. The series impedance goes on end 1, referred to its
voltage, and both ends carry their ratings.

In [4]:
xfmr_builder = PowerTransformerBuilder(network, container)
xfmr = xfmr_builder.create(mRID="xf_sub", name="substation")
xfmr_builder.add_connectivity(xfmr)
xfmr, _, high_end = xfmr_builder.add_power_transformer_end(
    xfmr, node="sourcebus", endNumber=1, connectionKind="D", BaseVoltage="bv_115kV"
)
xfmr, _, low_end = xfmr_builder.add_power_transformer_end(
    xfmr,
    node="subbus",
    endNumber=2,
    connectionKind="Yn",
    grounded=True,
    BaseVoltage="bv_12_47kV",
)
xfmr_builder.add_electrical(
    high_end,
    r=Qty(6.6, "ohm"),
    x=Qty(105, "ohm"),
    ratedS=Qty(10, "MVA"),
    ratedU=Qty(115, "kV"),
)
xfmr_builder.add_electrical(low_end, ratedS=Qty(10, "MVA"), ratedU=Qty(12.47, "kV"))

for end in (high_end, low_end):
    print(
        end.mRID,
        end.connectionKind.value,
        end.Terminal.ConnectivityNode.name,
        f"{end.ratedU.to('kV'):g} kV",
    )

xf_sub_1 D sourcebus 115 kV
xf_sub_2 Yn subbus 12.47 kV


## The feeder breaker

Switches connect two nodes. `normalOpen` is the switch's normal position, and `add_ssh(open=...)` its
position in this operating case.

In [5]:
breaker_builder = BreakerBuilder(network, container)
breaker = breaker_builder.create(mRID="brk_feeder", name="feeder breaker")
breaker_builder.add_connectivity(
    breaker,
    node1="subbus",
    node2="feeder_head",
    phases="ABC",
    normalOpen=False,
    BaseVoltage="bv_12_47kV",
)
breaker_builder.add_electrical(
    breaker, ratedCurrent=Qty(600, "A"), breakingCapacity=Qty(25, "kA")
)
breaker_builder.add_ssh(breaker, open=False)

print(
    breaker.mRID,
    f"normalOpen={breaker.normalOpen} open={breaker.open}",
    f"{breaker.ratedCurrent.to('A'):g} A",
)

brk_feeder normalOpen=False open=False 600 A


## The line

The line's impedance comes from a `PerLengthSequenceImpedance`, a per-length catalog entry that any number
of lines can share. The line itself has only its nodes and its length.

In [6]:
impedance_builder = PerLengthSequenceImpedanceBuilder(network, container)
impedance = impedance_builder.create(mRID="plz_336_acsr", name="336 ACSR")
impedance_builder.add_electrical(
    impedance, r=Qty(0.306, "ohm/mi"), x=Qty(0.627, "ohm/mi"), bch=Qty(5.4, "uS/mi")
)
impedance_builder.add_short_circuit(
    impedance, r0=Qty(0.775, "ohm/mi"), x0=Qty(1.98, "ohm/mi")
)

line_builder = ACLineSegmentBuilder(network, container)
line = line_builder.create(mRID="line_1", name="line_1")
line_builder.add_connectivity(
    line,
    node1="feeder_head",
    node2="bus_1",
    phases="ABC",
    length=Qty(2, "mi"),
    BaseVoltage="bv_12_47kV",
)
line_builder.add_per_length_impedance(line, PerLengthImpedance="plz_336_acsr")

r = line.PerLengthImpedance.r
print(f"{line.length.to('mi'):.1f} mi, r = {r.to('ohm/mi'):.3f} ohm/mi")

2.0 mi, r = 0.306 ohm/mi


## The load, capacitor bank and PV system

All three connect to `bus_1`. The capacitor's susceptance is worked out from its rated voltage and
reactive power. The PV system is a `PowerElectronicsConnection`, the inverter, with a `PhotoVoltaicUnit`
behind it. Generation is negative `p`, power flowing out of the equipment's terminal.

In [7]:
load_builder = EnergyConsumerBuilder(network, container)
load = load_builder.create(mRID="ld_1", name="load_1")
load_builder.add_connectivity(
    load,
    node="bus_1",
    phaseConnection="Y",
    phases="ABC",
    grounded=True,
    BaseVoltage="bv_12_47kV",
)
load_builder.add_ssh(load, p=Qty(1.5, "MW"), q=Qty(0.5, "MVAr"))

cap_builder = LinearShuntCompensatorBuilder(network, container)
cap = cap_builder.create(mRID="cap_1", name="cap_1")
cap_builder.add_connectivity(
    cap,
    node="bus_1",
    compensatorType="capacitor",
    phaseConnection="Y",
    phases="ABC",
    grounded=True,
    BaseVoltage="bv_12_47kV",
)
cap_builder.add_electrical(cap, nomU=Qty(12.47, "kV"), nomQ=Qty(600, "kVAr"))
cap_builder.add_ssh(cap, sections=1)

inverter_builder = PowerElectronicsConnectionBuilder(network, container)
inverter = inverter_builder.create(mRID="pv_1", name="pv_1")
inverter_builder.add_connectivity(
    inverter,
    node="bus_1",
    phaseConnection="Y",
    phases="ABC",
    BaseVoltage="bv_12_47kV",
)
inverter_builder.add_electrical(
    inverter, ratedS=Qty(500, "kVA"), ratedU=Qty(12.47, "kV")
)
inverter_builder.add_ssh(inverter, p=Qty(-400, "kW"), q=Qty(0, "kVAr"))

unit_builder = PhotoVoltaicUnitBuilder(network, container)
unit = unit_builder.create(mRID="pv_1_unit", name="pv_1")
unit_builder.add_electrical(
    unit, maxP=Qty(500, "kW"), PowerElectronicsConnection="pv_1"
)
unit_builder.add_ssh(unit, p=Qty(-400, "kW"))

print(f"capacitor bPerSection = {cap.bPerSection.to('uS'):.1f} µS")

capacitor bPerSection = 3858.5 µS


## What was built

Every object above is in `network.graph`, keyed by CIM class:

In [8]:
for cim_class, objects in sorted(
    network.graph.items(), key=lambda item: item[0].__name__
):
    print(f"{cim_class.__name__:32} {len(objects)}")

ACLineSegment                    1
ACLineSegmentPhase               3
BaseFrequency                    1
BaseVoltage                      2
Breaker                          1
ConnectivityNode                 4
EnergyConsumer                   1
EnergyConsumerPhase              3
EnergySource                     1
EnergySourcePhase                3
Feeder                           1
LinearShuntCompensator           1
LinearShuntCompensatorPhase      3
PerLengthSequenceImpedance       1
PhotoVoltaicUnit                 1
PowerElectronicsConnection       1
PowerElectronicsConnectionPhase  3
PowerTransformer                 1
PowerTransformerEnd              2
SwitchPhase                      3
Terminal                         10


## Save to CIM XML and read it back

`write_xml` writes the whole graph as CIM RDF/XML. Reloading it into a new model gets the same objects back,
with their associations and values.

In [9]:
os.makedirs("output", exist_ok=True)
write_xml(network, "output/feeder_from_scratch.xml")

roundtrip = FeederModel(
    container=cim.Feeder(mRID="fdr_demo"),
    connection=XMLFile(filename="output/feeder_from_scratch.xml"),
    distributed=False,
)
line = roundtrip.find_by_mRID("line_1")
node1, node2 = [
    terminal.ConnectivityNode.name
    for terminal in sorted(line.Terminals, key=lambda t: t.sequenceNumber)
]
load = roundtrip.find_by_mRID("ld_1")
length = line.length.to("mi")
print(f"line_1: {node1} - {node2}, {length:.1f} mi of {line.PerLengthImpedance.name}")
print(f"ld_1:   {load.p.to('kW'):g} kW, {load.q.to('kVAr'):g} kVAr")
print("objects:", sum(len(objects) for objects in roundtrip.graph.values()))

line_1: feeder_head - bus_1, 2.0 mi of 336 ACSR
ld_1:   1500 kW, 500 kVAr
objects: 47
